In [847]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import pandas as pd
pd.set_option('display.max_columns', 200)

In [808]:
df = pd.read_csv('../data/kaggle_b2_fraud_train_v3.csv')
df_test = pd.read_csv('../data/kaggle_b2_fraud_test_v3.csv')

## A. Qualité des données / nettoyage logique

### 1.1 Valeurs impossibles détectées

In [809]:
# suppression des âges négatifs
df = df[df['age'] >= 0]
df_test = df_test[df_test['age'] >= 0]

# suppression des dates de création de compte négatives
df = df[df["tenure_months"] >= 0]
df_test = df_test[df_test["tenure_months"] >= 0]

# suppression des revenus négatifs
df = df[df["annual_income_eur"] >= 0]
df_test = df_test[df_test["annual_income_eur"] >= 0]

# suppression des montants négatifs
df = df[df["avg_amount_30d_eur"] >= 0]
df_test = df_test[df_test["avg_amount_30d_eur"] >= 0]

### 1.2 Corrections de typologie nécessaires

In [810]:
# Conversions de types
for dataset in [df, df_test]:
    dataset['is_new_device'] = dataset['is_new_device'].astype('int64')
    dataset['postal_code'] = dataset['postal_code'].astype('string')
    dataset['days_since_last_login'] = dataset['days_since_last_login'].astype('int64')
    dataset['signup_source'] = dataset['signup_source'].astype('object')

### 1.3 suppression doublons inutiles

In [811]:
# suppression des lignes avec des customer_id en double
df = df.drop_duplicates(subset=['customer_id'], keep='first')
df_test = df_test.drop_duplicates(subset=['customer_id'], keep='first')

### 1.4 doublons stricts

In [812]:
df = df.drop_duplicates()
df_test = df_test.drop_duplicates()

## B. Data leakage connu

In [813]:
colonnes_a_supprimer = [
    'chargeback_resolution_time_days',
    'post_event_status_code'
]

df = df.drop(columns=colonnes_a_supprimer)
df_test = df_test.drop(columns=colonnes_a_supprimer)

## C. Identifiants et colonnes inutiles

In [814]:
# Suppression des colonnes avec trop de NaN (>90%)
cols_to_drop = ["legacy_partner_score", "partner_risk_indicator"]

df = df.drop(columns=cols_to_drop)
df_test = df_test.drop(columns=cols_to_drop)

In [815]:
# suppression de 680 lignes avec des customer_id en double ( très peu de changement donc variation)
df = df.drop_duplicates(subset=['customer_id'], keep='first')
df_test = df_test.drop_duplicates(subset=['customer_id'], keep='first')

In [816]:
# trop granulaire, inutile ou non exploitable
cols_to_drop = [
    "referrer_code",   # trop granulaire
    "postal_code",     # trop granulaire : risque overfitting
    "account_id",      # inutile
    "signup_date",      # pas d'informations facilement exploitable
    "city"             # trop granulaire
]

df = df.drop(columns=cols_to_drop)
df_test = df_test.drop(columns=cols_to_drop)

In [817]:
# Créer la colonne has_second_email (1 si secondary_email existe, 0 sinon)
df['has_second_email'] = df['secondary_email'].notna().astype(int)
df_test['has_second_email'] = df_test['secondary_email'].notna().astype(int)

# Supprimer la colonne secondary_email
df = df.drop(columns=['secondary_email'])
df_test = df_test.drop(columns=['secondary_email'])

In [818]:
# Créer les flags pour TOUTES les variables avec missingness informatif
for dataset in [df, df_test]:
    dataset['is_missing_last_ticket_subject'] = dataset['last_ticket_subject'].isna().astype(int)
    dataset['is_missing_max_amount_30d_eur'] = dataset['max_amount_30d_eur'].isna().astype(int)

In [819]:
## supprimer les colonnes de texte car on sait pas les traiter
text_cols = ["customer_note","last_ticket_subject"]

df = df.drop(columns=text_cols)
df_test = df_test.drop(columns=text_cols)

## D. Création de features basiques

In [820]:
# création de combinaison nouvelles et logiques

for dataset in [df, df_test]:
    # Interaction binaire × numérique
    dataset["is_new_device_x_num_devices"] = dataset["is_new_device"] * dataset["num_devices_30d"]

    # Interaction binaire × binaire
    dataset["is_vpn_x_ip_risk"] = dataset["is_vpn"] * dataset["ip_risk_z"]

## E. imputations ne dépendant pas de statistiques

In [821]:
df['max_amount_30d_eur'] = df['max_amount_30d_eur'].fillna(0)
df_test['max_amount_30d_eur'] = df_test['max_amount_30d_eur'].fillna(0)

In [822]:
df["ip_risk_z"] = df["ip_risk_z"].fillna(0)
df_test["ip_risk_z"] = df_test["ip_risk_z"].fillna(0)

## F : Nettoyage des variables catégorielles

In [823]:
## 
df.occupation.unique()

mapping = {
    'self employed': 'self_employed',
    'free lancer': 'freelancer',
    'freeelancer': 'freelancer',
    'public-sector': 'public_sector',
}

df['occupation'] = df['occupation'].str.strip().str.lower().replace(mapping)

In [824]:
df.payment_method.unique()

mapping_payment = {
    'ApplePay': 'apple_pay',
    'pay pal': 'paypal',
    'SEPA ': 'sepa',
    'googlepay': 'google_pay',
}

df['payment_method'] = df['payment_method'].str.strip().str.lower().replace(mapping_payment)

## Début : Suppression temporaire des features catégorielles à conseerver mais non encodés
## Je les rajouterai petit à petit

In [825]:
# Nombre total de lignes
n_rows = len(df)

# Calcul du nombre et du pourcentage de NaN
missing_df = pd.DataFrame({
    'nb_null': df.isnull().sum(),
    'percent_null': df.isnull().sum() / n_rows * 100
})

# Trier par % décroissant
missing_df = missing_df.sort_values(by='percent_null', ascending=False)
missing_df.head(8)

,nb_null,percent_null
region,38932,28.292988
credit_score,6882,5.001344
device_trust_z,5488,3.988285
occupation,4073,2.959965
is_vpn_x_ip_risk,4069,2.957058
merchant_category,2714,1.972341
customer_id,0,0.000000
tenure_months,0,0.000000


In [826]:
fill_values = {
    'occupation': 'Missing',
    'merchant_category': 'Missing',
    'last_ticket_subject': 'No_ticket',
    'customer_note': 'No_note'
}

df.fillna(value=fill_values, inplace=True)
df_test.fillna(value=fill_values, inplace=True)

# -----------------------------------------------
#                            TRAIN TEST SPLIT
# -----------------------------------------------

In [827]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from category_encoders import TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer

In [828]:
train, val = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42,
    stratify=df['target_is_fraud']
)

# df_test = fichier Kaggle sans target → on le garde tel quel
test = df_test.copy()

In [829]:
# 1. SAUVEGARDER customer_id AVANT TOUT
# ─────────────────────────────────────────
train_customer_ids = train['customer_id']
val_customer_ids = val['customer_id']
test_customer_ids = test['customer_id']

# Retirer customer_id des 3 datasets
train = train.drop(columns=['customer_id'])
val = val.drop(columns=['customer_id'])
test = test.drop(columns=['customer_id'])

In [830]:
# 1. SÉPARATION X ET Y
# On sépare les features de la target pour chaque dataset
# Le modèle a besoin de les avoir séparément
# ─────────────────────────────────────────
X_train = train.drop(columns=['target_is_fraud'])
y_train = train['target_is_fraud']

X_val = val.drop(columns=['target_is_fraud'])
y_val = val['target_is_fraud']

# Test → pas de target (vient de Kaggle)
X_test = test.copy()

Multicolinéarité

In [831]:
# Liste des colonnes à supprimer (VIF trop fort)
col_vif_trop_fort = [
    'terms_accepted_flag',
    'credit_score',
    'income_log',
    'avg_amount_30d_eur',
    'credit_score_norm',
    'income_estimate_alt_eur',
    'max_to_avg_ratio',
    'age',
    'tx_amount_total_30d_eur'
]

# Supprimer les colonnes dans train, val et test
for dataset in [train, val, test]:
    dataset.drop(columns=col_vif_trop_fort, inplace=True)

In [832]:

# device_trust_z → 0
# is_vpn_x_ip_risk → 0
for dataset in [train, val, test]:
    dataset['device_trust_z'] = dataset['device_trust_z'].fillna(0)
    dataset['is_vpn_x_ip_risk'] = dataset['is_vpn_x_ip_risk'].fillna(0)

### encodage des variables catégorielles

In [833]:
# Colonnes nominales (pas d'ordre) → OneHotEncoder
onehot_cols = [
    "signup_source", "os", "browser", "device_type", 
    "channel", "country", "payment_method", 
    "merchant_category", "occupation"
]

# Colonnes ordinales (avec ordre) → OrdinalEncoder
ordinal_cols = ["plan_type", "manual_review_result"]

ordinal_order = [
    ["basic", "standard", "premium", "enterprise"],  # plan_type
    ["approve", "review", "block"]                    # manual_review_result
]

### encodage des variables numériques

In [834]:
# Colonnes numériques
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()

# Retirer la target des colonnes numériques
num_cols = [col for col in num_cols if col != 'target_is_fraud']

### création des pipelines pour encodages

In [835]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

onehot_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

ordinal_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=ordinal_order,
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

In [837]:
# 2. PREPROCESSOR SANS PASSTHROUGH
# On retire le passthrough de la target puisqu'elle est déjà séparée
# remainder='drop' ignore toute colonne non déclarée
# ─────────────────────────────────────────
preprocessor = ColumnTransformer(transformers=[
    ('num',     numeric_pipeline,  num_cols),    # imputation médiane + scaling
    ('onehot',  onehot_pipeline,   onehot_cols), # imputation + OHE
    ('ordinal', ordinal_pipeline,  ordinal_cols), # imputation + OrdinalEncoder
])

In [838]:
preprocessor

,transformers,"[('num', ...), ('onehot', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


## gerer l'unbalancing sur le train

In [839]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline  # important ! pas sklearn

In [840]:
# ─────────────────────────────────────────
# 3. PIPELINE SMOTE
# On chaîne le preprocessor et SMOTE dans un seul pipeline
# imblearn Pipeline est nécessaire car sklearn ne supporte pas SMOTE
# ─────────────────────────────────────────
full_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42))
])

In [841]:
# 4. FIT + RESAMPLE SUR TRAIN UNIQUEMENT
# fit_resample : 
#   → apprend les statistiques du preprocessor (moyenne, std, catégories...)
#   → transforme X_train
#   → génère des exemples synthétiques avec SMOTE pour équilibrer les classes
# SMOTE ne s'applique JAMAIS sur val et test
# ─────────────────────────────────────────
X_train_res, y_train_res = full_pipeline.fit_resample(X_train, y_train)

In [842]:
# Récupérer les noms de colonnes
ohe_feature_names = preprocessor.named_transformers_['onehot']['encoder'].get_feature_names_out(onehot_cols).tolist()
all_cols = num_cols + ohe_feature_names + ordinal_cols

In [843]:
# 5. TRANSFORM SUR VAL ET TEST
# On applique UNIQUEMENT le preprocessor déjà fitté sur train
# Pas de SMOTE ici : on ne rééquilibre pas val et test
# Les statistiques apprises sur train sont appliquées telles quelles
# ─────────────────────────────────────────
X_val_res = preprocessor.transform(X_val)
X_test_res = preprocessor.transform(X_test)

In [845]:
# Recoller X et y avec les bons noms
train_final = pd.DataFrame(X_train_res, columns=all_cols)
train_final['target_is_fraud'] = y_train_res

val_final = pd.DataFrame(X_val_res, columns=all_cols)
val_final.insert(0, 'customer_id', val_customer_ids.values)
val_final['target_is_fraud'] = y_val.values

test_final = pd.DataFrame(X_test_res, columns=all_cols)
test_final.insert(0, 'customer_id', test_customer_ids.values)

In [850]:
test_final.head()

,customer_id,tenure_months,annual_income_eur,num_transactions_30d,max_amount_30d_eur,days_since_last_login,support_tickets_90d,chargebacks_12m,failed_payments_6m,device_trust_z,ip_risk_z,is_vpn,num_devices_30d,is_new_device,internal_signal_1,internal_signal_2,internal_signal_3,internal_signal_4,internal_signal_5,internal_signal_6,internal_signal_7,internal_signal_8,has_second_email,is_missing_last_ticket_subject,is_missing_max_amount_30d_eur,is_new_device_x_num_devices,is_vpn_x_ip_risk,signup_source_ads_search,signup_source_ads_social,signup_source_affiliate,signup_source_email_campaign,signup_source_organic,signup_source_referral,os_Android,os_Linux,os_Other,os_Windows,os_iOS,os_macOS,browser_Chrome,browser_Edge,browser_Firefox,browser_Opera,browser_Other,browser_Safari,device_type_desktop,device_type_iot_device,device_type_laptop,device_type_phone,device_type_tablet,channel_call_center,channel_mobile_app,channel_partner_api,channel_web,country_BE,country_CA,country_CH,country_DE,country_DZ,country_ES,country_FR,country_GB,country_IT,country_MA,country_NL,country_TN,country_US,payment_method_apple_pay,payment_method_applepay,payment_method_card,payment_method_crypto,payment_method_google_pay,payment_method_paypal,payment_method_sepa,merchant_category_Missing,merchant_category_digital_services,merchant_category_electronics,merchant_category_fashion,merchant_category_gaming,merchant_category_groceries,merchant_category_home,merchant_category_luxury,merchant_category_sports,merchant_category_travel,occupation_Missing,occupation_employee,occupation_executive,occupation_freelancer,occupation_public_sector,occupation_retired,occupation_self_employed,occupation_student,occupation_unemployed,plan_type,manual_review_result
0,CUST_E5RX1BC9II,1.966287,-0.446663,0.033253,1.926713,-0.380413,-0.892876,-0.223474,-0.592265,-1.206276,-1.973710,-0.294947,-0.644753,2.136200,0.644677,-0.790863,1.622840,-0.773814,-1.332833,-0.673874,0.963190,0.137696,-0.292831,-0.172361,-0.227808,1.007873,-0.001421,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,CUST_BHWIUKERGN,-0.728610,0.287023,-0.055504,-0.120455,-0.629794,0.225271,-0.223474,-0.592265,-1.031737,0.438447,-0.294947,-0.644753,-0.468121,-2.647382,0.884554,2.223573,-0.225887,0.389368,0.992496,1.558485,1.057458,-0.292831,-0.172361,-0.227808,-0.397252,-0.001421,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,CUST_EXT9NA4CHU,-0.897041,-0.036789,-0.173847,-0.175294,-0.380413,0.225271,-0.223474,-0.592265,0.956980,-0.107970,-0.294947,0.483942,-0.468121,-0.728071,-0.160780,0.130633,-0.415016,-1.026376,-1.158711,-0.334904,-0.151715,-0.292831,-0.172361,-0.227808,-0.397252,-0.001421,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
3,CUST_9FSJE5R1NY,0.113545,-0.272582,0.181181,0.144794,1.365251,-0.892876,-0.223474,1.103604,-1.082045,-0.792515,-0.294947,-0.644753,-0.468121,0.057853,-1.926928,0.834877,-1.413544,0.876541,0.475879,0.411600,0.235317,-0.292831,-0.172361,-0.227808,-0.397252,-0.001421,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,CUST_GDQXMODBED,-0.447892,-0.086817,-0.055504,0.993094,0.783363,-0.892876,-0.223474,-0

In [ ]:
# Exporter
train_final.to_csv('train_1.csv', index=False)
val_final.to_csv('val_1.csv', index=False)
test_final.to_csv('test_1.csv', index=False)